## Создание базы данных в Pandas

## Импорты библиотек

In [14]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import pymorphy3
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /home/stas/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /home/stas/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Массив для записи в его чанков, чанк это кусок лекции где слов чуть больше 500

In [2]:
array_of_chuncks = []

In [ ]:
# массив с названиями файлов в дериктории ./student_book_txt
array_of_books = [book for book in os.listdir('./student_book_txt')] 

NameError: name 'os' is not defined

In [ ]:
# алгоритм деления на чанки текста с ./student_book_txt
idx_chunk = 0
count_word = 0
for book in tqdm(array_of_books):
    with open(f'./student_book_txt/{book}', 'r', encoding='utf-8') as file:
        line = file.readline()
        while line:
            count_word += len(line.split())
            try:
                array_of_chuncks[idx_chunk] += line
            except IndexError as err:
                array_of_chuncks.append(line)
            if count_word > 500:
                idx_chunk += 1
                count_word = 0
            line = file.readline()
        idx_chunk += 1
        count_word = 0

        

NameError: name 'array_of_books' is not defined

In [5]:
array_of_chuncks

['кафедра «Математическое моделирование»\nпроф. П. Л. Иванков\nИнтегралы и дифференциальные уравнения\nконспект лекций\nдля студентов 1-го курса 2-го семестра\nспециальностей РЛ1,2,3,6, БМТ1,2\nЛекция 24\nОднородные системы линейных дифференциальных уравнений с по-\nстоянными коэффициентами. Характеристическое уравнение си-\nстемы. Построение общего решения по корням характеристического\nуравнения (вывод только для случая действительных и различных\nкорней).\nРассмотрим линейную однородную систему с постоянными коэффициентами\nn\nX\ny0 = a y , i = 1,...,n . (1)\ni ij j\nj=1\nВ матричной форме эта система запишется так:\nY0 = AY , (2)\nгде\n\uf8eb \uf8f6\na a ... a\n\uf8eb \uf8f6 11 12 1n\ny\n1 a a ... a\n\uf8ec 21 22 2n \uf8f7\nY = \uf8ed ... \uf8f8 , A = \uf8ec \uf8f7 .\n··················\n\uf8ed \uf8f8\ny\nn\na a ... a\nn1 n2 nn\nХарактеристическим уравнением системы (1) (или (2)) называется уравнение\n \na −λ a ... a\n 11 12 1n \n \na a −λ ... a\n 21 22 2n \n  = 0 .\n ·············

## Создание импровезированной бд с помощью pandas

In [ ]:
df = pd.DataFrame(array_of_chuncks, columns=['chunc_text']) 

In [ ]:
# из-за перевода в csv можно теперь работать с df с помощью read_csv
df = pd.read_csv("database.csv")

In [5]:
df.head()

,chunc_text
0,кафедра «Математическое моделирование» проф. П...
1,1 1 2 2 1 2 Т.к. коэффициенты характеристическ...
2,степенях x слева и справа. В результате получа...
3,−1 3−λ 1 2 Для определения компонент собстве...
4,кафедра «Математическое моделирование» проф. П...


In [6]:
# замена переходов на следующую строку на обычные пробелы
df['chunc_text'] = df['chunc_text'].str.replace('\n', ' ')

In [7]:
#  перевод в нижний регистр всех текстов
df['chunc_text'] = df['chunc_text'].str.casefold()

In [8]:
df.head()

,chunc_text
0,кафедра «математическое моделирование» проф. п...
1,1 1 2 2 1 2 т.к. коэффициенты характеристическ...
2,степенях x слева и справа. в результате получа...
3,−1 3−λ 1 2 для определения компонент собстве...
4,кафедра «математическое моделирование» проф. п...


In [16]:
# загружаю stop_words (предлоги и приставки)
russian_stop_words = set(stopwords.words('russian'))

## Токенизация с помощью nltk и лэмматизация pymorphy3

In [18]:
morph = pymorphy3.MorphAnalyzer()
new_data_for_data_frame = []
cache = {}
for idx in range(len(df)):
    chunk = df.iloc[idx]['chunc_text']
    chunk = word_tokenize(chunk, language='russian')
    tokenize_chunk = [w for w in chunk if w.isalpha() and w not in russian_stop_words]
    for i, word in enumerate(tokenize_chunk):
        if word not in cache:
            cache[word] = morph.parse(word)[0].normal_form
        tokenize_chunk[i] = cache[word]
    new_data_for_data_frame.append(tokenize_chunk)

In [19]:
df['tokinaze_chunk'] = pd.Series(new_data_for_data_frame)

In [21]:
df.iloc[2]['tokinaze_chunk']

['степень',
 'x',
 'слева',
 'справа',
 'результат',
 'получаться',
 'система',
 'линейный',
 'однородный',
 'браический',
 'уравнение',
 'который',
 'удовлетворять',
 'компонент',
 'столбцы',
 'c',
 'c',
 'решать',
 'система',
 'получить',
 'r',
 'линейно',
 'независимый',
 'решение',
 'исходный',
 'система',
 'ренциальный',
 'уравнение',
 'известно',
 'комплексный',
 'корень',
 'являться',
 'вещественный',
 'уравнение',
 'ственный',
 'коэффициент',
 'распадаться',
 'пара',
 'комплексно',
 'сопрячь',
 'корень',
 'один',
 'тот',
 'кратность',
 'β',
 'один',
 'такой',
 'пара',
 'кратность',
 'r',
 'описать',
 'выше',
 'процедура',
 'следовать',
 'применить',
 'комплексный',
 'случай',
 'один',
 'корень',
 'α',
 'iβ',
 'затем',
 'отделить',
 'получить',
 'комплекснозначный',
 'решение',
 'вещественный',
 'мнимый',
 'часть',
 'проделаввсеэтодлякаждогокорняхарактеристическогоуравнение',
 'даментальный',
 'система',
 'решение',
 'исходный',
 'система',
 'дифференциальный',
 'уравнение',
 'п

In [5]:
import pandas as pd
corpus = list(pd.read_csv("../database.csv")['lemma_text'])
isinstance(corpus[0], str)

True

In [ ]:
"дисперсия" in corpus[852]
print(corpus[852])

распределение закон распределение быть говорить характеристика распределение математический ожидание случайный величина определение математический ожидание средний значение mx дискретный случайный величина x называть сумма произведение значение x cлучайный величина i ность p который случайный величина принимать значение i i x p i i i множество возможный значение cлучайный величина x счётный предполагаться i i ряд определять математический ожидание сходиться абсолютно противный случай рят математический ожидание случайный величина x существовать ник пусть прямой расположить система материальный точка масса p p пусть i i i x координата точка центр масса система иметь координата i x p x p i i i i i i x p p i i i i i совпадать математический ожидание mx случайный величина пример пусть x число выпасть очко подбрасывание игральный кость p i i определение математический ожидание средний значение mx непрерывный случайный величина называть интеграл xp x dx предполагаться x dx несобственный инте

In [11]:
df = pd.read_csv("../database.csv")['lemma_text']
sum(df.isna())

4